<a href="https://colab.research.google.com/github/quickklearner/Sequence_Marketing/blob/main/Sequence_Marketing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **DATA 6400 Final Project Robust Fake Review Detection with Large Language Models: A Prompt Rewording and Review Length Analysis**

## Jose Barral, Victoria González, Arjun Moitra & Ooreoluwa Ayoola
## April 10th, 2026


Acknowledgments:
* We used ChatGPT for some syntax guidance and all instances of this are commented appropriately below.
* All other programming, text explanations and analysis are our own.

## 1. Imports and Setup

First, we need to download an older version of numpy in order to be able to use the hugging face dataset.

In [ ]:
!pip -q uninstall -y datasets
!pip -q install datasets==2.18.0
!pip install -U "numpy<2.0" transformers accelerate evaluate bitsandbytes huggingface_hub

In [ ]:
!pip -q install -U sentencepiece

Libraries Needed:

In [ ]:
#Dataset Import
import datasets
from datasets import load_dataset

#Data Handling
import pandas as pd
import numpy as np
import gc
import os
import re

#Tokenization
from collections import Counter
from transformers import AutoTokenizer

#Modeling
from transformers import AutoModelForSequenceClassification, AutoModelForCausalLM
from transformers import TrainingArguments, Trainer, BitsAndBytesConfig
import torch

#Progress
from tqdm.auto import tqdm

#Metrics
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

#Visualisation
import matplotlib.pyplot as plt

#HuggingFace Hub for saving the models
from huggingface_hub import login

Then, we load the hugging face dataset.

In [ ]:
print(datasets.__version__)

#Load of hugging face dataset
ds = load_dataset("difraud/difraud", "product_reviews", trust_remote_code=True)
ds

As shown, the dataset is already split into training, validation, and test sets, containing 16,776, 2,097, and 2,098 observations, respectively.

## 2. Data Quality Check

In [ ]:
train_df = pd.DataFrame(ds["train"])
val_df   = pd.DataFrame(ds["validation"])
test_df  = pd.DataFrame(ds["test"])

train_df.head()

In [ ]:
# Any null values?
print(train_df.isnull().sum())

# Class balance?
print(train_df["label"].value_counts())
print(train_df["label"].value_counts(normalize=True))

# Basic length
train_df["text_length"] = train_df["text"].str.len()
print(train_df["text_length"].describe())

The dataset shows no missing values in either the text or label variables, indicating that it is complete and ready for modeling.

In terms of class distribution, the dataset is perfectly balanced, with 8,393 instances labeled as 1 and 8,383 as 0. This corresponds to proportions of 50.03% and 49.97%, respectively, ensuring no class imbalance issues that could bias the model.

Regarding text length, there is significant variability across reviews. The average length is approximately 362 characters, with a median of 233, indicating a right-skewed distribution. The shortest review contains 62 characters, while the longest reaches 15,957 characters. The interquartile range (154 to 381 characters) shows that most reviews are relatively short, although the presence of very long outliers suggests that truncation during tokenization could be necessary.

## 3. Length Analysis

We analyze the token lengths to define a balanced categorization of reviews into short, medium, and long groups.

In [ ]:
#For this, we will use the Roberta base tokenizer
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

def count_tokens(example):
    tokens = tokenizer.tokenize(example["text"])
    example["n_tokens"] = len(tokens) + 2
    return example

ds = ds.map(count_tokens)

In [ ]:
train_lengths = ds["train"]["n_tokens"]

q1 = np.quantile(train_lengths, 0.33)
q2 = np.quantile(train_lengths, 0.66)

print("Q1:", q1)
print("Q2:", q2)

Then, reviews are categorized as follows: short (≤ 42 tokens), medium (43–71 tokens), and long (> 71 tokens).

Then, we create the length groups based on the previously identified quantile thresholds.

In [ ]:
def assign_length_group(example):
    if example["n_tokens"] <= q1:
        example["length_group"] = "short"
    elif example["n_tokens"] <= q2:
        example["length_group"] = "medium"
    else:
        example["length_group"] = "long"
    return example

ds = ds.map(assign_length_group)

In [ ]:
print("Train:", Counter(ds["train"]["length_group"]))
print("Validation:", Counter(ds["validation"]["length_group"]))
print("Test:", Counter(ds["test"]["length_group"]))

## 4. Models

### 4.1. Roberta Model

We will use Roberta as the baseline model.

#### 4.1.1. Model architecture

We tokenize the input text using the RoBERTa tokenizer. Truncation and padding are applied to a maximum sequence length of 512 tokens to ensure uniform input size across all samples.

In [ ]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

ds_tokenized = ds.map(tokenize, batched=True)

After tokenization, the original text column is removed, and the label column is renamed to match the expected input format (labels). The dataset is then converted to PyTorch tensors for model training.

In [ ]:
ds_tokenized = ds_tokenized.remove_columns(["text"])
ds_tokenized = ds_tokenized.rename_column("label", "labels")

ds_tokenized.set_format("torch")

A pre-trained RoBERTa-base model is loaded for sequence classification, with a binary output layer (num_labels = 2) corresponding to the FAKE/REAL classification task.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2
)

Model performance is evaluated using accuracy, precision, recall, and F1-score. Predictions are obtained by selecting the class with the highest logit score. Given the nature of the task, recall is used as the main metric for model selection to prioritize the detection of fake reviews.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", pos_label=1, zero_division=0
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

Evaluation and model saving are performed at the end of each epoch, and the best model is selected based on recall. Weight decay is applied to improve generalization.

In [ ]:
training_args = TrainingArguments(
    output_dir="./roberta_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="recall",
    greater_is_better=True,
    report_to="none",
    disable_tqdm=True
)

Then, the Hugging Face Trainer is used to manage the training and evaluation process by integrating the model, training arguments, datasets, and evaluation metrics into a unified pipeline.

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_tokenized["train"],
    eval_dataset=ds_tokenized["validation"],
    compute_metrics=compute_metrics
)

#### 4.1.2. Model training

In [ ]:
trainer.train()

#### 4.1.3. Model evaluation

In [ ]:
test_metrics = trainer.evaluate(eval_dataset=ds_tokenized["test"])
print(test_metrics)

In [ ]:
# Test predictions
pred_output = trainer.predict(ds_tokenized["test"])
pred_labels = pred_output.predictions.argmax(axis=1)
true_labels = pred_output.label_ids

# Unify with original dataset
test_results_df = pd.DataFrame(ds["test"])
test_results_df["pred_label"] = pred_labels
test_results_df["true_label"] = true_labels

# Metrics by group
group_metrics = []

for group in ["short", "medium", "long"]:
    subset = test_results_df[test_results_df["length_group"] == group]

    y_true = subset["true_label"]
    y_pred = subset["pred_label"]

    group_metrics.append({
        "length_group": group,
        "n": len(subset),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "recall": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "f1": f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    })

group_metrics_df = pd.DataFrame(group_metrics)
print(group_metrics_df)

Since the primary objective of this study is to maximize recall, we focus on the model’s ability to correctly identify deceptive reviews. The results show that recall varies across review length groups. Medium-length reviews achieved the highest recall (0.829), indicating that the model successfully identifies a large proportion of deceptive reviews in this category. In comparison, recall decreases for long reviews (0.729) and short reviews (0.721). This suggests that reviews of moderate length provide the most informative signals for detecting deception, while very short reviews lack sufficient context and very long reviews may introduce additional noise.

#### 4.1.4. Saving of results

In [ ]:
from huggingface_hub import login
login()  # enter your HuggingFace token when prompted

# note results and model have been saved to hugging face profile quickklearner

In [ ]:
# Save trained RoBERTa model to HuggingFace Hub
trainer.save_model("./roberta_fake_review_model")
tokenizer.save_pretrained("./roberta_fake_review_model")

from transformers import AutoModelForSequenceClassification, AutoTokenizer
model_to_push = AutoModelForSequenceClassification.from_pretrained("./roberta_fake_review_model")
tok_to_push   = AutoTokenizer.from_pretrained("./roberta_fake_review_model")

model_to_push.push_to_hub("quickklearner/roberta-fake-review")
tok_to_push.push_to_hub("quickklearner/roberta-fake-review")

print("Model pushed to HuggingFace Hub.")
print("Anyone can now load it with:")
print('  AutoModelForSequenceClassification.from_pretrained("quickklearner/roberta-fake-review")')

### 4.2. Mistral Zero-Shot Model


#### 4.2.1. Model setup

We use the Mistral-7B-Instruct model for zero-shot classification. The model is loaded using the Hugging Face Transformers library, along with its corresponding tokenizer. To optimize performance, half-precision (float16) is used when a GPU is available, and the model is automatically distributed across devices using device_map="auto".

In [ ]:
model_id = "mistralai/Mistral-7B-Instruct-v0.3"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

#### 4.2.1. Prompt design

The task is framed as a zero-shot classification problem, where each review is embedded into a prompt instructing the model to classify it as FAKE or REAL. Different prompt variations are tested to evaluate their impact on performance.

In [ ]:
prompt_A = """Classify this product review as FAKE or REAL.

A REAL review usually reflects genuine product experience, including concrete details, natural wording, or personal usage.
A FAKE review often relies on vague praise, exaggerated enthusiasm, or promotional language without clear evidence of real use.

Output exactly one word:
FAKE
or
REAL

Review:
{review}
"""

prompt_B = """Classify this product review as FAKE or REAL.

Consider whether it reflects genuine product experience or vague promotional language.

Output exactly one word:
FAKE
or
REAL

Review:
{review}
"""

prompt_C = """Classify this product review as FAKE or REAL.

Use cues such as specific experience, vague praise, and promotional wording.

Output exactly one word:
FAKE
or
REAL

Review:
{review}
"""

In [ ]:
prompts = {
    "prompt_A": prompt_A,
    "prompt_B": prompt_B,
    "prompt_C": prompt_C
}

#### 4.2.2. Inference and Output parsing

During inference, each review is embedded into a structured prompt and passed to the model using a chat-based format. The model generates a short textual response, which is constrained to a maximum of a few tokens to ensure concise outputs.

Since the model is generative, the raw output must be processed to extract a valid prediction. Therefore, the generated response is decoded, trimmed, and parsed to obtain a clean label (FAKE or REAL), ensuring consistency across predictions.

In [ ]:
def classify_review(review_text, prompt_template):

    messages = [
        {
            "role": "system",
            "content": (
                "You are a product review classifier. "
                "Classify each review as FAKE or REAL. "
                "Output only one word. "
                "Allowed outputs: FAKE or REAL. "
                "Do not explain. "
                "Do not add punctuation. "
                "Do not add extra text."
            )
        },
        {
            "role": "user",
            "content": prompt_template.format(review=review_text)
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    outputs = model.generate(
        **inputs,
        max_new_tokens=3,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    input_len = inputs["input_ids"].shape[1]
    response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

    return response.strip()

In [ ]:
def parse_prediction(text):
    if text is None:
        return None

    text = text.strip().upper()

    # Caso exacto ideal
    if text == "FAKE":
        return 1
    if text == "REAL":
        return 0

    # Buscar etiquetas aisladas
    matches = re.findall(r"\b(FAKE|REAL)\b", text)

    if not matches:
        return None

    # Tomar la última, que suele ser la respuesta final
    last_label = matches[-1]

    if last_label == "FAKE":
        return 1
    elif last_label == "REAL":
        return 0

    return None

#### 4.2.3. Prediction Generation

In [ ]:
test_df = pd.DataFrame(ds["test"]).head(100).copy()

In [ ]:
for prompt_name, prompt_template in prompts.items():

    preds = []
    raw_outputs = []

    for review in test_df["text"]:

        output = classify_review(review, prompt_template)

        raw_outputs.append(output)
        preds.append(parse_prediction(output))

    test_df[f"raw_{prompt_name}"] = raw_outputs
    test_df[f"pred_{prompt_name}"] = preds

#### 4.2.4. Model evaluation

In [ ]:
for prompt_name in prompts.keys():

    valid = test_df[test_df[f"pred_{prompt_name}"].notna()]

    recall = recall_score(
        valid["label"],
        valid[f"pred_{prompt_name}"],
        pos_label=1,
        zero_division=0
    )

    print(prompt_name, recall)

In [ ]:
results = []

for prompt_name in prompts.keys():

    for group in ["short","medium","long"]:

        subset = test_df[
            (test_df["length_group"] == group) &
            (test_df[f"pred_{prompt_name}"].notna())
        ]

        rec = recall_score(
            subset["label"],
            subset[f"pred_{prompt_name}"],
            pos_label=1,
            zero_division=0
        )

        results.append({
            "prompt": prompt_name,
            "length_group": group,
            "n": len(subset),
            "recall": rec
        })

results_df = pd.DataFrame(results)
print(results_df)

###### 4.2.4.1. Prompt Disagreement

To assess the consistency of the model, we compute the disagreement rate between different prompt formulations. A disagreement occurs when at least one prompt produces a different prediction for the same review.

In [ ]:
test_df["prompt_disagreement"] = (
    (test_df["pred_prompt_A"] != test_df["pred_prompt_B"]) |
    (test_df["pred_prompt_A"] != test_df["pred_prompt_C"]) |
    (test_df["pred_prompt_B"] != test_df["pred_prompt_C"])
)

In [ ]:
change_rate = test_df["prompt_disagreement"].mean()

print("Prompt disagreement rate:", change_rate)

In [ ]:
test_df.groupby("length_group")["prompt_disagreement"].mean()

In [ ]:
test_df[test_df["prompt_disagreement"]].head(5)

##4.3. Qwen Model

#### 4.3.1. Model setup

Qwen3.5-9B is loaded in 4-bit NF4 quantization using BitsAndBytes to reduce GPU memory usage while preserving model quality. A custom tokenizer is also loaded and configured with a pad token.

In [ ]:
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

QWEN_MODEL = "Qwen/Qwen3.5-9B"

quant_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

print(f"Loading tokenizer: {QWEN_MODEL}")
qwen_tok = AutoTokenizer.from_pretrained(QWEN_MODEL)

if qwen_tok.pad_token is None:
    qwen_tok.pad_token = qwen_tok.eos_token

print(f"Loading model (4-bit NF4): {QWEN_MODEL}")
qwen_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL,
    quantization_config=quant_cfg,
    device_map="auto",
    low_cpu_mem_usage=True,
)

qwen_model.eval()
print("\n Model loaded successfully.")

#### 4.3.2. Prompt design

Three prompts (A, B, C) are defined with varying levels of detail. A system message enforces a structured output format ending with `FINAL_LABEL: FAKE` or `FINAL_LABEL: REAL`.

In [ ]:
SYSTEM_MSG = (
    "You are an expert classifier for fake product reviews. "
    "You may think internally, but your final response must end with exactly one line in this format: "
    "FINAL_LABEL: FAKE or FINAL_LABEL: REAL. "
    "Do not end with anything else after that final line."
)

PROMPT_A = """Classify this product review as FAKE or REAL.

A REAL review usually reflects genuine product experience, including concrete details, natural wording, or personal usage.
A FAKE review often relies on vague praise, exaggerated enthusiasm, or promotional language without clear evidence of real use.

Output exactly one word:
FAKE
or
REAL

Review:
{review}
"""

PROMPT_B = """Classify this product review as FAKE or REAL.

Consider whether it reflects genuine product experience or vague promotional language.

Output exactly one word:
FAKE
or
REAL

Review:
{review}
"""

PROMPT_C = """Classify this product review as FAKE or REAL.

Use cues such as specific experience, vague praise, and promotional wording.

Output exactly one word:
FAKE
or
REAL

Review:
{review}
"""

PROMPTS = {
    "prompt_A": PROMPT_A,
    "prompt_B": PROMPT_B,
    "prompt_C": PROMPT_C,
}

print(f" {len(PROMPTS)} prompts defined.")

#### 4.3.3. Inference and output parsing

The `parse_output` function extracts FAKE/REAL from the model's response using regex, with fallbacks for synonyms. The `classify_review` function formats the prompt using the chat template and runs inference.

In [ ]:
def parse_output(text: str):
    if text is None:
        return None

    t = text.strip().upper()

    # Best case: explicit final label
    m = re.search(r"FINAL_LABEL:\s*(FAKE|REAL)", t)
    if m:
        return 1 if m.group(1) == "FAKE" else 0

    # Fallback: last FAKE/REAL mention anywhere
    hits = re.findall(r"\b(FAKE|REAL)\b", t)
    if hits:
        return 1 if hits[-1] == "FAKE" else 0

    if any(w in t for w in ["DECEPTIVE", "FRAUDULENT", "FABRICATED", "INAUTHENTIC"]):
        return 1
    if any(w in t for w in ["GENUINE", "AUTHENTIC", "LEGITIMATE"]):
        return 0

    return None


def classify_review(review_text: str, prompt_template: str,
                    max_input_length: int = 1024, max_new_tokens: int = 120):
    prompt = prompt_template.format(review=review_text)

    messages = [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user", "content": prompt},
    ]

    formatted = qwen_tok.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    try:
        inputs = qwen_tok(
            formatted,
            return_tensors="pt",
            truncation=True,
            max_length=max_input_length
        ).to(qwen_model.device)

        with torch.no_grad():
            out = qwen_model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=qwen_tok.eos_token_id,
            )

        new_tokens = out[0][inputs["input_ids"].shape[1]:]
        raw = qwen_tok.decode(new_tokens, skip_special_tokens=True).strip()
        parsed = parse_output(raw)

        return raw, parsed

    except Exception as e:
        print(f"[ERROR] {e}")
        return None, None

#### 4.3.4. Prediction generation

A stratified sample of 100 reviews is drawn from the test set (33 short, 33 medium, 34 long). Each prompt is run across the sample and checkpoints are saved every 200 reviews.

In [ ]:
# Prepare stratified test sample for Qwen
test_qwen = pd.DataFrame(ds["test"])

# Assign length groups using the same thresholds from section 3
test_qwen["n_tokens"] = ds["test"]["n_tokens"]
test_qwen["length_group"] = ds["test"]["length_group"]

short_df  = test_qwen[test_qwen["length_group"] == "short"].sample(n=33, random_state=42)
medium_df = test_qwen[test_qwen["length_group"] == "medium"].sample(n=33, random_state=42)
long_df   = test_qwen[test_qwen["length_group"] == "long"].sample(n=34, random_state=42)

eval_df = pd.concat([short_df, medium_df, long_df]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Evaluating {len(eval_df)} reviews.")
print(eval_df["length_group"].value_counts())

for prompt_name, prompt_template in PROMPTS.items():
    print(f"\n{'='*60}")
    print(f"Prompt: {prompt_name}")
    print(f"{'='*60}")

    raw_col  = f"raw_{prompt_name}"
    pred_col = f"pred_{prompt_name}"
    raws, preds, failed = [], [], 0

    for i, review in enumerate(tqdm(eval_df["text"], desc=prompt_name)):
        raw, label = classify_review(review, prompt_template)
        raws.append(raw)
        preds.append(label)
        if label is None:
            failed += 1

    eval_df[raw_col]  = raws
    eval_df[pred_col] = preds
    print(f"✓ Done — total failed: {failed}/{len(eval_df)}")

#### 4.3.5. Model evaluation

Overall metrics (accuracy, precision, recall, F1) are computed per prompt, followed by a breakdown by review length group.

In [ ]:
print("\n" + "="*60)
print("OVERALL METRICS — Qwen3.5-9B Zero-Shot (3 prompts)")
print("="*60)

overall_rows = []

for prompt_name in PROMPTS:
    pred_col = f"pred_{prompt_name}"
    valid = eval_df.dropna(subset=[pred_col]).copy()
    valid[pred_col] = valid[pred_col].astype(int)

    y_true = valid["label"]
    y_pred = valid[pred_col]

    row = {
        "prompt":    prompt_name,
        "n_valid":   len(valid),
        "n_failed":  len(eval_df) - len(valid),
        "accuracy":  round(accuracy_score(y_true, y_pred), 4),
        "precision": round(precision_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
        "recall":    round(recall_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
        "f1":        round(f1_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
    }
    overall_rows.append(row)

    print(f"\n── {prompt_name} ──")
    print(classification_report(y_true, y_pred, target_names=["Real", "Fake"], zero_division=0))

overall_df = pd.DataFrame(overall_rows)
print("\n=== Summary Table ===")
print(overall_df.to_string(index=False))

In [ ]:
group_rows = []

for prompt_name in PROMPTS:
    pred_col = f"pred_{prompt_name}"
    for group in ["short", "medium", "long"]:
        sub = eval_df[
            (eval_df["length_group"] == group) &
            (eval_df[pred_col].notna())
        ].copy()

        if len(sub) == 0:
            continue

        sub[pred_col] = sub[pred_col].astype(int)
        y_true = sub["label"]
        y_pred = sub[pred_col]

        group_rows.append({
            "prompt":       prompt_name,
            "length_group": group,
            "n":            len(sub),
            "accuracy":     round(accuracy_score(y_true, y_pred), 4),
            "precision":    round(precision_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
            "recall":       round(recall_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
            "f1":           round(f1_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
        })

group_df = pd.DataFrame(group_rows)
print("=== Metrics by Length Group ===")
print(group_df.to_string(index=False))

###### 4.3.5.1. Prompt disagreement

We compute how often the three prompts disagree on the same review, overall and broken down by length group.

In [ ]:
pred_cols = [f"pred_{p}" for p in PROMPTS]
complete  = eval_df.dropna(subset=pred_cols).copy()

for c in pred_cols:
    complete[c] = complete[c].astype(int)

complete["prompt_disagreement"] = complete[pred_cols].nunique(axis=1) > 1

overall_dis = complete["prompt_disagreement"].mean()
print(f"Overall prompt disagreement rate: {overall_dis:.3f}")

dis_by_len = complete.groupby("length_group")["prompt_disagreement"].mean()
print("\nDisagreement by length group:")
print(dis_by_len.to_string())

print("\nSample reviews where prompts disagree:")
display_cols = ["text", "label", "length_group"] + pred_cols
print(complete[complete["prompt_disagreement"]][display_cols].head(5).to_string())

#### 4.3.6. Visualisations

In [ ]:
best_prompt = overall_df.loc[overall_df["f1"].idxmax(), "prompt"]
print(f"Best prompt by F1: {best_prompt}")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, prompt_name in zip(axes, PROMPTS):
    pred_col = f"pred_{prompt_name}"
    valid = eval_df.dropna(subset=[pred_col]).copy()
    valid[pred_col] = valid[pred_col].astype(int)

    cm   = confusion_matrix(valid["label"], valid[pred_col])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Real", "Fake"])
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    title = prompt_name + (" ★ Best" if prompt_name == best_prompt else "")
    ax.set_title(title, fontsize=11)

plt.suptitle("Qwen3.5-9B Zero-Shot — Confusion Matrices (3 Prompts)", fontsize=13, y=1.03)
plt.tight_layout()
plt.show()

#### 4.3.7. Saving results

Results are saved to HuggingFace datasets (via CSV) so the team can access them without needing anyone's Google Drive.

In [ ]:
from huggingface_hub import login
login()  # enter your HuggingFace token when prompted

overall_df.to_csv("/content/qwen_overall_metrics.csv", index=False)
group_df.to_csv("/content/qwen_group_metrics.csv", index=False)
complete.to_csv("/content/qwen_all_predictions.csv", index=False)

print(" Qwen results saved locally.")

## 5. Comparative Results